## 补码与位掩码：nand2tetris 里的两个硬事实

> 2026-10-22 · 来源：nand2tetris 项目复习（modules/408/NAND2TETRIS.md）
- Hack 是 **16 位补码整数**，没有浮点。补码不对称：`-32768` 取负仍是 `-32768`（溢出回自身），`Math.abs` 必须单独处理这个边界
- 屏幕内存 `16384–24575` 共 8K 字，**1 bit = 1 像素**：512×256 像素、每行 32 字 → 由 `x/16` 定位字、`x%16` 定位位
- **改一个像素必须「读-改-写整个 16 位字」**：`v | mask` 置黑、`v & ~mask` 置白；只写一位会擦掉同行相邻 15 个像素
- 由此推出：水平线与竖直线算法必须分开写 —— 同行 16 个像素在同一字里（可批量算掩码），竖直线要跨 32 个字（行间距 32 字）


## CSAPP datalab：位运算的五件武器（408·数据表示）

> 2026-10-22 · 来源：CSAPP 项目复习（`modules/408/CSAPP.md`） §7.1（实践仓 `learning-408/csapp/datalab/bits.c`）
datalab 只允许 `! ~ & ^ | + << >>`（无 `-`、`*`、`if`、`?:`），常量只能 0–255，int 为 32 位补码、`>>` 是算术右移。13 道题全靠五件武器拼：
1. **条件掩码**：`mask = !!x << 31 >> 31`（x≠0 → 全 1；x=0 → 全 0），于是 `if` 变成 `(mask & y) | (~mask & z)`
2. **取负 / 减法**：`-x` 写 `~x + 1`；`x - y` 写 `x + ~y + 1`
3. **取符号位**：`x >> 31`（非负 → 0，负 → -1）
4. **判相等**：`!(a ^ b)`
5. **拼大常量**：`m = 0xAA; m |= m << 8; m |= m << 16;` → 0xAAAAAAAA

**408 对应**：原/反/补/移码、溢出判断、移位、IEEE 754 —— 组成原理选择题年年考，也是大题浮点计算的地基。
**边界**：datalab 的 `dlc`/`btest` 是 Linux x86-64 ELF，macOS 上跑不了，只能读/写代码。


## IEEE 754 的平滑衔接：为什么非规格化的 E 取 1 - Bias

> 2026-10-22 · 来源：CSAPP 项目复习（`modules/408/CSAPP.md`） §7.1.5
单精度布局：`s(1) | exp(8) | frac(23)`，Bias = 127。
- 规格化（`exp != 0`）：`E = exp - 127`，`M = 1.frac`
- 非规格化（`exp == 0`）：`E = 1 - 127 = -126`，`M = 0.frac`
- `exp == 0xFF`：`frac == 0` 为 ±∞，否则 NaN

**为什么是 `1 - Bias` 而不是 `-Bias`**（算证据，自己验一遍）：
- 最小规格化数：`exp=1, frac=0` → `1.0 × 2^(1-127) = 2^-126`
- 最大非规格化数：`exp=0, frac=0x7FFFFF` → `0.111…1₂ × 2^-126 = 2^-126 - 2^-149`
- 二者相差恰好一个最小步长 `2^-149`，**中间不留缝** → 逐级下溢（gradual underflow）才成立

**三个必须背的边界**：最小规格化 `0x00800000`、最大非规格化 `0x00400000`、最小非规格化 `0x00000001`；`floatPower2` 里 `x > 127` → +∞（`0xFF << 23`，不能写 0x7F800000，常量超限），`x < -149` → 0。
